In [3]:
import pandas as pd
from pathlib import Path

# from posydon.popsyn.binarypopulation import BinaryPopulation
# from posydon.binary_evol.binarystar import BinaryStar
# from posydon.binary_evol.singlestar import SingleStar
from posydon.popsyn.synthetic_population import Population
# from posydon.popsyn.synthetic_population import PopulationRunner
# import astropy.units as u

# import os
# import shutil
# from posydon.config import PATH_TO_POSYDON

# from POSYDONHRDiagramModule import HR_Diagram
# import matplotlib.pyplot as plt
# import random as rand 

from collections import Counter

DataPath = Path().resolve().parent / 'Data'


In [4]:
cols = ['time', 'step_names', 'state', 'event', 'S1_state', 'S2_state', 'S1_mass', 'S2_mass', 'orbital_period']
finCols = [
    'orbital_period_f',
    'eccentricity_f',
    'state_f',

    'S1_state_f',
    'S2_state_f',
    
    'S2_mass_f',
    'S2_log_R_f',
    'S2_log_L_f',


    'S1_mass_f',
    'S1_log_R_f',
    'S1_log_L_f'

 ]

initCols = [
    'orbital_period_i',
    'eccentricity_i',
    'state_i',

    'S2_state_i',
    'S2_mass_i',
    'S2_log_R_i',

    'S1_state_i',
    'S1_mass_i',
    'S1_log_R_i'
 ]

In [5]:
pop = Population(str(DataPath / 'bhSolSubpop.h5'))

In [6]:
len(pop)

39161

In [7]:
pop.calculate_formation_channels()

Formation channels already exist in the parsed population file!
Channels will be overwriten


In [8]:
Counter(pop.formation_channels['channel'])

Counter({'ZAMS_oRLO1_CC1_oRLO2_CC2_END': 13728,
         'ZAMS_oRLO1_CC1_oRLO2_CC2_maxtime_END': 7867,
         'ZAMS_CC1_oRLO2_CC2_maxtime_END': 4133,
         'ZAMS_CC1_oRLO2_CC2_END': 3981,
         'ZAMS_oRLO1_CC1_oRLO2_oCE2_oMerging2_END': 2854,
         'ZAMS_oRLO1_CC1_END': 1198,
         'ZAMS_CC1_CC2_END': 1131,
         'ZAMS_oRLO1_CC1_oRLO2_oCE2_CC2_END': 1002,
         'ZAMS_CC1_CC2_maxtime_END': 801,
         'ZAMS_CC1_oRLO2_oCE2_oMerging2_END': 375,
         'ZAMS_oRLO1_CC1_oRLO2_oCE2_CC2_maxtime_END': 323,
         'ZAMS_oCE1_CC1_oRLO2_CC2_maxtime_END': 251,
         'ZAMS_oRLO1-contact_CC1_oRLO2_oCE2_oMerging2_END': 247,
         'ZAMS_oRLO1_CC1_oRLO2_oCE2_END': 220,
         'ZAMS_oRLO1_CC1_CC2_END': 184,
         'ZAMS_oRLO1-contact_CC1_END': 141,
         'ZAMS_oRLO1_CC1_CC2_maxtime_END': 121,
         'ZAMS_CC1_END': 86,
         'ZAMS_oCE1_CC1_END': 86,
         'ZAMS_oCE1_CC1_oRLO2_oCE2_oMerging2_END': 74,
         'ZAMS_oCE1_CC1_oRLO2_CC2_END': 55,
         'ZAMS

In [9]:
solBH_oneline_df = pop.oneline.select()
solBH_history_df = pop.history.select()

In [10]:
sol_bhInitMask = (
    ((solBH_history_df['S1_state'] == 'BH') & (solBH_history_df['S2_state'] == 'H-rich_Core_H_burning')) |
    ((solBH_history_df['S2_state'] == 'BH') & (solBH_history_df['S1_state'] == 'H-rich_Core_H_burning'))
) & (solBH_history_df['event'] != 'END')

sol_bhMaskNext = sol_bhInitMask.groupby(level=0).shift(1, fill_value=False)

sol_bhMask = sol_bhInitMask | sol_bhMaskNext

specStatedf = solBH_history_df[sol_bhMask] ### makes a df with the row with the sol-bh and the row after. used for finding the lifespan of the systems 

sol_bhMaskFirst = pd.Series(False, index = sol_bhMask.index)
sol_bhMaskFirst = ~sol_bhMaskFirst.index.duplicated(keep='first')

firstAndsol_bh = sol_bhInitMask | sol_bhMaskFirst

# def isMW(group): ### this is wayyyyy too narrow of a process... it does work though!
#     return (group['time'].iloc[0] < 10e9) & (group['time'].iloc[1] > 10e9)
# results = solBH_history_df[sol_bhMask].groupby(level=0).apply(isMW)

def findStatDur(group):
        return (group['time'].iloc[1] - group['time'].iloc[0])


In [11]:
solBH_history_df[firstAndsol_bh][cols]

,time,step_names,state,event,S1_state,S2_state,S1_mass,S2_mass,orbital_period
binary_index,,,,,,,,,
0,1.354058e+10,initial_cond,detached,ZAMS,H-rich_Core_H_burning,H-rich_Core_H_burning,35.594066,20.634703,86.368549
0,1.354658e+10,step_SN,detached,NaN,BH,H-rich_Core_H_burning,11.742339,20.766340,773.934254
1,9.269310e+08,initial_cond,detached,ZAMS,H-rich_Core_H_burning,H-rich_Core_H_burning,21.815063,21.446482,4.949364
1,9.365578e+08,step_SN,detached,NaN,BH,H-rich_Core_H_burning,2.862360,27.696843,13.464326
1,9.365578e+08,step_detached,initial_RLOF,NaN,BH,H-rich_Core_H_burning,2.862360,27.479589,13.464326
...,...,...,...,...,...,...,...,...,...
39206,9.352825e+09,initial_cond,detached,ZAMS,H-rich_Core_H_burning,H-rich_Core_H_burning,20.373868,8.859054,154.288032
39206,9.362281e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,3.746892,9.131854,0.941797
39206,9.362281e+09,step_detached,initial_RLOF,NaN,BH,H-rich_Core_H_burning,3.746892,9.135707,0.941797


In [12]:
sol_bh_lifespan_list = specStatedf.groupby(level=0).apply(findStatDur)
sol_bh_FormationTime_List = solBH_history_df[firstAndsol_bh].groupby(level=0).apply(findStatDur)

In [13]:
sol_bh_lifespan_list.describe()

count    3.916100e+04
mean     7.712614e+06
std      2.035345e+07
min      0.000000e+00
25%      1.116906e+06
50%      3.391723e+06
75%      8.622863e+06
max      1.373802e+09
dtype: float64

In [14]:
sol_bh_FormationTime_List.describe()

count    3.916100e+04
mean     5.937467e+06
std      2.474277e+07
min      3.200919e+06
25%      4.150988e+06
50%      5.172376e+06
75%      6.617883e+06
max      2.467654e+09
dtype: float64

In [42]:
tolerance = 1e9

sol_bhMaskMWdur = ((
    ((solBH_history_df['S1_state'] == 'BH') & (solBH_history_df['S2_state'] == 'H-rich_Core_H_burning')) |
    ((solBH_history_df['S2_state'] == 'BH') & (solBH_history_df['S1_state'] == 'H-rich_Core_H_burning'))
    ) & (solBH_history_df['event'] != 'END')
        & (solBH_history_df['time'] > 10e9 - tolerance)
        & (solBH_history_df['time'] < 10e9 + tolerance)
    & (solBH_history_df['state'] == 'detached')
)

In [56]:
lowPorbSolBH = solBH_history_df[sol_bhMaskMWdur][solBH_history_df[sol_bhMaskMWdur]['orbital_period'] < 3][cols]

In [57]:
lowPorbSolBH

,time,step_names,state,event,S1_state,S2_state,S1_mass,S2_mass,orbital_period
binary_index,,,,,,,,,
247,9.005453e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,2.720699,7.515792,2.597624
582,1.095540e+10,step_SN,detached,NaN,BH,H-rich_Core_H_burning,5.858426,11.333348,1.460274
703,1.071638e+10,step_SN,detached,NaN,BH,H-rich_Core_H_burning,3.925841,23.076179,2.895749
2029,9.467351e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,5.446302,22.059192,2.399101
2086,9.179265e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,2.578924,8.765645,2.581370
...,...,...,...,...,...,...,...,...,...
37932,9.790867e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,3.623130,7.206539,2.092048
38056,9.258820e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,3.623154,7.688547,1.565752
38116,9.836158e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,4.368231,21.563922,2.993418


In [58]:
pop.export_selection(lowPorbSolBH.index.to_list(), str(DataPath / 'bhSol_10Gyr_LowPorb_Subpop.h5'), append=False, overwrite=True)

In [59]:
filtPop = Population(str(DataPath / 'bhSol_10Gyr_LowPorb_Subpop.h5'))

In [60]:
len(filtPop)

70

In [61]:
filtPop.history.select()[cols]

,time,step_names,state,event,S1_state,S2_state,S1_mass,S2_mass,orbital_period
binary_index,,,,,,,,,
0,8.995911e+09,initial_cond,detached,ZAMS,H-rich_Core_H_burning,H-rich_Core_H_burning,21.649254,7.025360,13.252380
0,9.005453e+09,step_HMS_HMS,detached,CC1,stripped_He_Core_C_depleted,H-rich_Core_H_burning,6.748006,7.515792,2.207866
0,9.005453e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,2.720699,7.515792,2.597624
0,9.005453e+09,step_detached,initial_RLOF,NaN,BH,H-rich_Core_H_burning,2.720699,7.536774,2.597624
0,9.005453e+09,step_end,initial_RLOF,END,BH,H-rich_Core_H_burning,2.720699,7.536774,2.597624
...,...,...,...,...,...,...,...,...,...
69,9.362273e+09,step_CE,detached,NaN,stripped_He_non_burning,H-rich_Core_H_burning,7.457188,9.061668,1.493836
69,9.362281e+09,step_detached,detached,CC1,stripped_He_Core_He_depleted,H-rich_Core_H_burning,7.396282,9.131854,1.514407
69,9.362281e+09,step_SN,detached,NaN,BH,H-rich_Core_H_burning,3.746892,9.131854,0.941797
